# baseline GPU/CPU 비교 — 43개와 47개가 갈린 이유 확인

**질문**: 원래 V4(취약 후보 43개)와 재현 코드 01(47개)의 B가 다른 이유가 **GPU/CPU 차이**인가?

**하는 일**
1. 01과 같은 전처리·변수·설정으로 B를 **CPU 1번, GPU 2번** 학습합니다. 시드는 모두 42예요.
2. 각 B를 원래 V4 CSV와 비교합니다. 입력(S·N·평시·휴무·특보)이 같은지도 같이 봅니다.
3. 02 판정 코드를 각 V4로 돌려 취약 후보 수를 셉니다.

**준비**
- 런타임 → 런타임 유형 변경 → **T4 GPU**
- Drive `big_contest/data`: 카드 데이터1, `강남_춘천_기상데이터_정제*.csv`, `특보데이터_강남춘천_일별발효*.csv` (01과 같음)
- 원래 V4 CSV 경로를 0번 셀 `ORIG_V4`에 적기
- `02_취약판정_유형나누기.ipynb`를 Drive `big_contest` 아래 아무 곳에 두기 (자동으로 찾음)

결과는 `results/GPU_CPU비교/`에 저장되고, **기존 `outputs/38_최종baseline_V4.csv`는 건드리지 않습니다.** 전체 소요는 약 5~10분입니다(카드 데이터 읽기와 학습 3번).


In [ ]:
# 0. 설정 — 경로만 확인하세요
from pathlib import Path
import io, contextlib, json
ROOT = Path('/content/drive/MyDrive/big_contest')
try:
    from google.colab import drive; drive.mount('/content/drive')
except ImportError:
    pass

# ① 43개가 나온 원래 V4 CSV (아래 목록을 보고 경로를 정확히 적어 주세요)
ORIG_V4 = ROOT / 'outputs' / '38_최종baseline_V4_원래.csv'
# ② 02 재현 노트북 위치 (자동으로 찾음, 못 찾으면 직접 적기)
CODE_02 = None
# ③ 결과 저장 폴더 (기존 outputs 파일을 덮어쓰지 않도록 따로 저장)
OUT_CMP = ROOT / 'results' / 'GPU_CPU비교'; OUT_CMP.mkdir(parents=True, exist_ok=True)
# ④ 돌릴 조합: (이름, 장치). GPU를 두 번 돌리는 건 GPU끼리도 매번 같은지 보려는 것
RUNS = [('CPU', None), ('GPU_1', 'cuda'), ('GPU_2', 'cuda')]
RUN_02 = True        # 02 판정까지 돌려 후보 수를 셀지 (한 번에 몇 분씩 걸림)

print('찾은 V4 파일 (수정 시각순):')
import os, time as _t
for p in sorted(ROOT.rglob('38_최종baseline_V4*.csv'), key=lambda p: p.stat().st_mtime):
    print('  ', _t.strftime('%Y-%m-%d %H:%M', _t.localtime(p.stat().st_mtime)), p)
assert ORIG_V4.exists(), f'ORIG_V4 경로를 위 목록에서 골라 고쳐 주세요: {ORIG_V4}'
if CODE_02 is None:
    hits = sorted(ROOT.rglob('02_취약판정_유형나누기.ipynb'), key=lambda p: p.stat().st_mtime)
    assert hits or not RUN_02, '02_취약판정_유형나누기.ipynb를 Drive의 big_contest 아래에 올리거나 CODE_02에 경로를 적어 주세요'
    CODE_02 = hits[-1] if hits else None
print('원래 V4:', ORIG_V4); print('02 노트북:', CODE_02)


## 1. 01과 같은 설정

In [ ]:
import os, re, glob, time, warnings, subprocess, sys
import numpy as np, pandas as pd
from IPython.display import display
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
try: import xgboost
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
import xgboost as xgb
print('xgboost', xgb.__version__, '/ pandas', pd.__version__, '/ numpy', np.__version__)

# ---- 경로 ----
DATA_DIR = str(ROOT / 'data')
def find(pattern):
    hits = sorted(glob.glob(f'{DATA_DIR}/**/{pattern}', recursive=True))
    if not hits: raise FileNotFoundError(f'{DATA_DIR}에서 {pattern}을 찾지 못했습니다')
    return hits[0]
CARD_PATH = find('신한카드_빅콘테스트2026_데이터1*')
WEATHER_PATH = find('강남_춘천_기상데이터_정제*.csv')
try: FLAG_PATH = find('특보데이터_강남춘천_일별발효_등급*.csv')
except FileNotFoundError: FLAG_PATH = find('특보데이터_강남춘천_일별발효*.csv')
print(CARD_PATH, WEATHER_PATH, FLAG_PATH, sep='\n')

# ---- 전처리 설정 ----
REGION_MAP = {'서울 강남구': '강남(서울동남권)', '강원 춘천시': '춘천'}
EXCLUDE_RY = ['세금공과금', 'ZZ_나머지']     # 법인(SEX_CCD)도 제외
MIN_NONZERO = 0.8      # 칸(지역×업종×시간대)의 매출>0 날 비율이 이 값 미만이면 희소 칸으로 제외
RAIN_MAX = 10          # 평시 = 폭염·호우 특보 없음 & 일강수 < 10mm
CLOSED_SHARE = 0.5     # 휴무 칸 = 그 칸·그 요일(공휴일은 따로)의 평시 날 중 매출 0 비율 ≥ 0.5

# ---- 달력 설정 ----
HOLI = pd.to_datetime(['2025-08-15', '2025-10-03', '2025-10-05', '2025-10-06', '2025-10-07', '2025-10-08',
                       '2025-10-09', '2025-12-25'])          # 2025 하반기 공휴일(대체공휴일 포함)
CHUSEOK_PRE = ('2025-09-22', '2025-10-02')
YEAR_END = ('2025-12-22', '2025-12-31')
XMAS_EVE = pd.Timestamp('2025-12-24')
SPECIAL_DAYS = pd.to_datetime(['2025-10-02', '2025-10-03', '2025-10-04', '2025-10-05', '2025-10-06', '2025-10-07',
                               '2025-10-08', '2025-10-09', '2025-12-24', '2025-12-25'])   # 검증에서 따로 볼 특수일(학습엔 그대로 씀)

# ---- 교차 적합 ----
N_FOLDS, BUFFER = 5, 3

# ---- 주기 달력을 넣는 업종: (지역 앞글자, 업종명에 포함된 글자) ----
CYCLE_INDS = [('강남', '유치원'), ('춘천', '유치원')]

# ---- 공휴일 → 일요일 처리 업종 ----
HOLI_SUN_ALL_REGIONS = ['강남']
HOLI_SUN_LIST = [('춘천', u) for u in ['LPG가스', '동물병원', '약국', '일반병원', '자동차용품', '종합병원', '주유소',
                                     '중고품판매점', '컴퓨터/소프트웨어', '학원/학습지', '화장품']]

# ---- 모델 ----
try: gpu = subprocess.run('nvidia-smi', capture_output=True).returncode == 0
except FileNotFoundError: gpu = False
XGB_PARAMS = dict(objective='reg:tweedie', tweedie_variance_power=1.5,
                  n_estimators=1000, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.8, colsample_bytree=1.0,
                  enable_categorical=True, max_cat_to_onehot=1, tree_method='hist', random_state=42)
SCALE = 1e4            # 목표값 = 매출 / 1만 원
print('GPU 있음:', gpu)
if not gpu: print('⚠ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU로 바꾸고 다시 실행하세요. (지금은 CPU만 돌립니다)')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)

## 2. 카드 → 칸 집계 (01과 같음)

In [ ]:
card = pd.read_csv(CARD_PATH, sep='\t', encoding='cp949', dtype={'TA_YMD': str, 'TIME_GB': str})
n0 = len(card)
card['TS_AT'] = pd.to_numeric(card['TS_AT'], errors='coerce')
card['USE_CNT'] = pd.to_numeric(card['USE_CNT'], errors='coerce')
card['날짜'] = pd.to_datetime(card['TA_YMD'], format='%Y%m%d')
card = card[(card['SEX_CCD'] != '법인') & (~card['MCT_RY_CD'].isin(EXCLUDE_RY))]
card['대상'] = card['MCT_SGG_CD'].map(REGION_MAP)
print(f'원본 {n0:,}행 → 법인·세금공과금·ZZ_나머지 제외 후 {len(card):,}행 / 지역 매핑 실패 {card["대상"].isna().sum():,}행')
agg = card.dropna(subset=['대상']).groupby(['날짜', '대상', 'MCT_RY_CD', 'TIME_GB'])[['TS_AT', 'USE_CNT']].sum()
del card

DATES = pd.date_range(agg.index.get_level_values(0).min(), agg.index.get_level_values(0).max())
grid = pd.MultiIndex.from_product([DATES, sorted(agg.index.get_level_values(1).unique()),
                                   sorted(agg.index.get_level_values(2).unique()), sorted(agg.index.get_level_values(3).unique())],
                                  names=['날짜', '대상', '업종', 'slot'])
P = agg.reindex(grid, fill_value=0).rename(columns={'TS_AT': 'S', 'USE_CNT': 'N'}).reset_index()
nz = P.groupby(['대상', '업종', 'slot'])['S'].apply(lambda s: (s > 0).mean())
keep = nz[nz >= MIN_NONZERO].index
P = P.set_index(['대상', '업종', 'slot']).loc[keep].reset_index()
SLOTS = sorted(P['slot'].unique(), key=lambda s: int(re.findall(r'\d+', s)[0]))
print(f'기간 {DATES.min().date()} ~ {DATES.max().date()} ({len(DATES)}일)')
print(f'남은 칸 {len(keep):,} / {len(nz):,}, 업종 수:', P.groupby('대상')['업종'].nunique().to_dict(), '/ 시간대:', SLOTS)

## 3. 특보·기상·달력·휴무 (01과 같음)

In [ ]:
fl = pd.read_csv(FLAG_PATH, encoding='utf-8-sig', parse_dates=['날짜'])
warn_col = next((c for c in fl.columns if '경보' in c), None)
fl['등급'] = np.where(fl[warn_col].fillna(0) > 0, 2, np.where(fl['발효'] > 0, 1, 0)) if warn_col else (fl['발효'] > 0).astype(int)
G = fl.pivot_table(index=['대상', '날짜'], columns='카테고리', values='등급', aggfunc='max', fill_value=0).reset_index()
for c in ['폭염', '호우', '한파', '대설']:
    if c not in G: G[c] = 0
G = G.rename(columns={'폭염': '폭염등급', '호우': '호우등급'})
G['한파대설'] = (G['한파'] > 0) | (G['대설'] > 0)

P = P.merge(G[['대상', '날짜', '폭염등급', '호우등급', '한파대설']], on=['대상', '날짜'], how='left')
P[['폭염등급', '호우등급']] = P[['폭염등급', '호우등급']].fillna(0).astype(int)
cold = P['한파대설'].fillna(False).astype(bool)
print('한파·대설로 제외한 날:', P.loc[cold].groupby('대상')['날짜'].nunique().to_dict())
P = P[~cold].drop(columns='한파대설')
dd = P.drop_duplicates(['대상', '날짜'])
print('폭염일(주의보만/경보):', dd.groupby('대상')['폭염등급'].apply(lambda s: f'{(s == 1).sum()}/{(s == 2).sum()}').to_dict(),
      '/ 호우일:', dd.groupby('대상')['호우등급'].apply(lambda s: int((s > 0).sum())).to_dict())

In [ ]:
W = pd.read_csv(WEATHER_PATH, encoding='utf-8-sig', parse_dates=['일시'])
W['날짜'] = W['일시'].dt.normalize(); W['hour'] = W['일시'].dt.hour
starts = [int(re.findall(r'\d+', s)[0]) for s in SLOTS]
W['slot'] = W['hour'].map(lambda h: SLOTS[max(i for i, s in enumerate(starts) if s <= h)])
WD_ = W.groupby(['대상', '날짜']).agg(일최고기온=('기온(°C)', 'max'), 일강수=('강수량(mm)', 'sum')).reset_index()
WS_ = W.groupby(['대상', '날짜', 'slot']).agg(시간대최고기온=('기온(°C)', 'max'), 시간대강수=('강수량(mm)', 'sum'),
                                             시간대강수시간=('강수량(mm)', lambda s: int((s > 0).sum()))).reset_index()
P = P.merge(WD_, on=['대상', '날짜'], how='left').merge(WS_, on=['대상', '날짜', 'slot'], how='left')
print('기상 결측 비율:', P[['일최고기온', '일강수', '시간대최고기온']].isna().mean().round(4).to_dict())

In [ ]:
P['gkey'] = P['대상'] + '|' + P['업종']
P['skey'] = P['gkey'] + '#' + P['slot']
P['요일'] = P['날짜'].dt.dayofweek
P['주말'] = P['요일'].ge(5).astype(int)
P['공휴일'] = P['날짜'].isin(HOLI).astype(int)
P['daytype'] = np.where(P['공휴일'].eq(1), 7, P['요일'])
P['추석직전'] = P['날짜'].between(*CHUSEOK_PRE).astype(int)
P['평시'] = P['폭염등급'].eq(0) & P['호우등급'].eq(0) & P['일강수'].fillna(0).lt(RAIN_MAX)
P['fold'] = P['날짜'].dt.isocalendar().week.astype(int) % N_FOLDS
P = P.sort_values(['gkey', 'slot', '날짜']).reset_index(drop=True)
dd = P.drop_duplicates(['대상', '날짜'])
print('평시(학습) 날 수 — 월별'); display(dd[dd['평시']].groupby(['대상', dd['날짜'].dt.month]).size().unstack(fill_value=0))

In [ ]:
zr = P[P['평시']].groupby(['skey', 'daytype'])['S'].apply(lambda s: (s == 0).mean()).rename('평시0비율')
P = P.join(zr, on=['skey', 'daytype'])
P['휴무'] = P['평시0비율'].fillna(0).ge(CLOSED_SHARE)
P = P.drop(columns='평시0비율')
print(f"휴무 칸 {P['휴무'].sum():,} ({P['휴무'].mean():.2%})")

In [ ]:
cal = pd.DataFrame({'날짜': pd.date_range(P['날짜'].min() - pd.Timedelta(days=5), P['날짜'].max() + pd.Timedelta(days=5))})
off = (cal['날짜'].dt.dayofweek.ge(5) | cal['날짜'].isin(HOLI)).to_numpy()
blk, pos = np.zeros(len(cal), int), np.zeros(len(cal), int)
i = 0
while i < len(cal):
    if not off[i]: i += 1; continue
    j = i
    while j + 1 < len(cal) and off[j + 1]: j += 1
    blk[i:j + 1] = j - i + 1
    if j - i + 1 >= 3: pos[i] = 1; pos[j] = 3; pos[i + 1:j] = 2
    i = j + 1
cal['연휴위치'] = pos
cal['연휴전날'] = (~off & (np.r_[blk[1:], 0] >= 3)).astype(int)
cal['연휴다음날'] = (~off & (np.r_[0, blk[:-1]] >= 3)).astype(int)
d = cal['날짜'].dt
cal['월초'] = d.day.le(3).astype(int)
cal['월말'] = (d.days_in_month - d.day).le(6).astype(int)
cal['연말'] = cal['날짜'].between(*YEAR_END).astype(int)
cal['성탄이브'] = cal['날짜'].eq(XMAS_EVE).astype(int)
P = P.merge(cal, on='날짜', how='left').sort_values(['gkey', 'slot', '날짜']).reset_index(drop=True)

chk = P.drop_duplicates('날짜').set_index('날짜')
show_ = chk[(chk[['연휴위치', '연휴전날', '연휴다음날', '성탄이브']].sum(axis=1) > 0) | chk['추석직전'].eq(1)]
print('[확인] 연휴·명절·연말 표시가 있는 날'); display(show_[['공휴일', '추석직전', '연휴위치', '연휴전날', '연휴다음날', '연말', '성탄이브']].assign(요일=show_.index.day_name().str[:3]))

In [ ]:
cal2 = pd.DataFrame({'날짜': pd.date_range(P['날짜'].min().replace(day=1), P['날짜'].max() + pd.offsets.MonthEnd(0))})
d = cal2['날짜'].dt
cal2['영업일'] = ~(d.dayofweek.ge(5) | cal2['날짜'].isin(HOLI))
ym = d.to_period('M')
cal2['일자'] = d.day
cal2['영업일순번'] = cal2.groupby(ym)['영업일'].cumsum().astype(int)
cal2['남은영업일'] = (cal2.groupby(ym)['영업일'].transform('sum') - cal2['영업일순번']).astype(int)
cal2['월내주차'] = ((d.day - 1) // 7 + 1).astype(int)
CYC_CAL = ['일자', '영업일순번', '남은영업일', '월내주차']
P = P.merge(cal2[['날짜'] + CYC_CAL], on='날짜', how='left').sort_values(['gkey', 'slot', '날짜']).reset_index(drop=True)

GK = P['gkey'].unique()
CYC_G = sorted(g for g in GK if any(g.split('|')[0].startswith(r) and p in g.split('|')[1] for r, p in CYCLE_INDS))
print('주기 달력 업종:', CYC_G)
if len(CYC_G) != len(CYCLE_INDS): print('⚠ 지정한 업종 중 데이터에 없는 업종이 있습니다(희소 칸으로 빠졌을 수 있음)')

In [ ]:
SUN_G = sorted(g for g in GK if any(g.startswith(r) for r in HOLI_SUN_ALL_REGIONS)
               or any(g.split('|')[0].startswith(r) and g.split('|')[1] == u for r, u in HOLI_SUN_LIST))
miss = [u for r, u in HOLI_SUN_LIST if not any(g.split('|')[0].startswith(r) and g.split('|')[1] == u for g in GK)]
print(f'일요일 처리 업종 {len(SUN_G)}개 (강남 {sum(g.startswith("강남") for g in SUN_G)}, 춘천 {sum(g.startswith("춘천") for g in SUN_G)})')
if miss: print('⚠ 데이터에 없는 춘천 업종:', miss)

## 4. 변수와 학습 함수 (01과 같음, 장치만 바꿀 수 있게)

In [ ]:
def fold_masks(f, k):
    dates = f['날짜'].values.astype('datetime64[D]'); te = (f['fold'] == k).values
    near = np.zeros(len(f), bool)
    for off_ in range(-BUFFER, BUFFER + 1):
        near |= np.isin(dates, np.unique(dates[te]) + np.timedelta64(off_, 'D'))
    return ~near, te          # (학습 가능 행 = 예측 주 ±3일 밖, 예측 행)
FOLD_P = {k: fold_masks(P, k) for k in range(N_FOLDS)}
PLAIN = P['평시'].to_numpy()

def mad_ratio(x):
    m = x.median(); return (x - m).abs().median() / m if m > 0 else np.nan
def vol_feats(trn):
    g = P.loc[trn, ['skey', 'S']].groupby('skey')['S']
    T = pd.DataFrame({'과거중앙값': np.log1p(g.median()), 'MAD비율': g.apply(mad_ratio),
                      '매출0비율': g.apply(lambda x: (x == 0).mean()), '관측날수': g.size().astype(float)})
    out = T.reindex(P['skey']); out.index = P.index; out['관측날수'] = out['관측날수'].fillna(0)
    return out

CAL = ['요일', '주말', '공휴일', '월초', '월말', '추석직전', '연휴위치', '연휴전날', '연휴다음날', '연말', '성탄이브']
X0 = pd.DataFrame({'대상': P['대상'].astype('category'), '업종': P['업종'].astype('category'), '시간대': P['slot'].astype('category')}, index=P.index)
for c in CAL: X0[c] = P[c].astype(int)
sun = P['공휴일'].eq(1).to_numpy() & P['gkey'].isin(SUN_G).to_numpy()
X0.loc[sun, '요일'] = 6; X0.loc[sun, '주말'] = 1
print(f'일요일로 바꾼 행 {sun.sum():,}개')
cm = P['gkey'].isin(CYC_G).to_numpy()
XC = pd.DataFrame({f'주기_{c}': np.where(cm, P[c].astype(float), np.nan) for c in CYC_CAL}, index=P.index)

SW = P['S'].to_numpy(float) / SCALE

def fit_B(device):
    params = dict(XGB_PARAMS)
    if device: params['device'] = device
    B = np.full(len(P), np.nan); t0 = time.time()
    for k, (free, te) in FOLD_P.items():
        trn = free & PLAIN
        X = pd.concat([X0, vol_feats(trn), XC], axis=1)
        m = xgb.XGBRegressor(**params).fit(X.loc[trn], SW[trn])
        B[te] = m.predict(X.loc[te]) * SCALE
    assert X.shape[1] == 22 and np.isfinite(B).all()
    print(f'  학습 완료 ({device or "cpu"}, {time.time() - t0:.0f}초)')
    return np.clip(B, 0, None)


## 5. CPU 1번, GPU 2번 학습

In [ ]:
# 6. 조합마다 B 학습 → CSV 저장 (이름: 38_V4_<조합>.csv)
COLS = ['대상', '업종', 'gkey', 'slot', '날짜', 'S', 'N', 'B', 'B_하루합', 'y', '휴무', '평시', '특수일', '폭염등급', '호우등급',
        '일최고기온', '일강수', '시간대최고기온', '시간대강수', '시간대강수시간', '요일', '공휴일', 'fold']
P['특수일'] = P['날짜'].isin(SPECIAL_DAYS)
V4_FILES = {}
for name, dev in RUNS:
    if dev == 'cuda' and not gpu:
        print(f'{name}: GPU 없음 → 건너뜀'); continue
    print(f'{name}:')
    Q = P.copy(); Q['B'] = fit_B(dev)
    Q['B_하루합'] = Q.groupby(['gkey', '날짜'])['B'].transform('sum')
    Q['y'] = (Q['S'] - Q['B']) / Q['B_하루합'].replace(0, np.nan)
    path = OUT_CMP / f'38_V4_{name}.csv'
    Q[COLS].to_csv(path, index=False, encoding='utf-8-sig'); V4_FILES[name] = path
    print('  저장:', path)


## 6. B 비교

In [ ]:
# 7. B 비교 — 원래 V4와, 그리고 조합끼리
K = ['gkey', 'slot', '날짜']
def load(p): return pd.read_csv(p, encoding='utf-8-sig', parse_dates=['날짜'])
ORIG = load(ORIG_V4)
TABS = {'원래': ORIG, **{n: load(p) for n, p in V4_FILES.items()}}

def compare(a, b):
    m = TABS[a].merge(TABS[b], on=K, how='outer', suffixes=('_a', '_b'), indicator=True)
    both = m[m['_merge'] == 'both']
    same_in = all((both[f'{c}_a'] == both[f'{c}_b']).all() for c in ['S', 'N', '평시', '휴무', '폭염등급', '호우등급'])
    rel = (both['B_b'] - both['B_a']) / both['B_a'].where(both['B_a'] > 0)
    return {'비교': f'{a} vs {b}', '행 일치': (m['_merge'] == 'both').all(),
            '입력(S·N·평시·휴무·특보) 일치': same_in,
            'B 완전히 같은 칸(%)': round((np.abs(both['B_b'] - both['B_a']) <= 1e-6 * both['B_a'].abs().clip(lower=1)).mean() * 100, 2),
            'B 차이 5%(%)': round(rel.quantile(.05) * 100, 2), 'B 차이 95%(%)': round(rel.quantile(.95) * 100, 2),
            'B 최대 차이(%)': round(rel.abs().max() * 100, 2)}

pairs = [('원래', n) for n in V4_FILES] + [(a, b) for i, a in enumerate(V4_FILES) for b in list(V4_FILES)[i + 1:]]
B_CMP = pd.DataFrame([compare(a, b) for a, b in pairs]).set_index('비교')
display(B_CMP)


## 7. 취약 후보 수 비교

In [ ]:
# 8. 02 판정을 각 V4로 돌려 후보 수 비교 (출력은 숨김)
def run_02(v4_path, out_dir):
    nb = json.load(open(CODE_02, encoding='utf-8'))
    cells = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']
    stop = next(i for i, s in enumerate(cells) if s.startswith('# 6.'))
    out_dir.mkdir(parents=True, exist_ok=True)
    ns = {'__name__': '__main__'}
    import matplotlib; matplotlib.use('Agg')
    with contextlib.redirect_stdout(io.StringIO()):
        for i, src in enumerate(cells[:stop + 1]):
            if i == 0:
                assert "V4_PATH = find('38_최종baseline_V4.csv')" in src and "OUT = ROOT/'results'/'유형나누기'" in src
                src = src.replace("V4_PATH = find('38_최종baseline_V4.csv')", f"V4_PATH = Path(r'{v4_path}')")
                src = src.replace("OUT = ROOT/'results'/'유형나누기'", f"OUT = Path(r'{out_dir}')")
            ns['display'] = lambda *a, **k: None
            exec(compile(src, f'02_cell{i}', 'exec'), ns)
            ns['plt'].close('all') if 'plt' in ns else None
    return pd.read_excel(out_dir / '유형나누기_결과.xlsx', sheet_name=None)

if RUN_02:
    RES = {}
    for name, path in [('원래', ORIG_V4), *V4_FILES.items()]:
        t0 = time.time(); RES[name] = run_02(path, OUT_CMP / f'02_{name}')
        print(f'{name}: 02 완료 ({time.time() - t0:.0f}초)')
    key = ['대상', '종류', '업종']
    base = RES['원래']['취약후보_유형'][key]
    rows = []
    for name, r in RES.items():
        c = r['취약후보_유형'][key]
        cnt = c.groupby(['대상', '종류']).size()
        mg = base.assign(o=1).merge(c.assign(n=1), on=key, how='outer')
        rows.append({'V4': name, **{f'{a[:2]} {b}': int(cnt.get((a, b), 0)) for a, b in sorted(cnt.index)}, '합계': len(c),
                     '원래와 겹침': int((mg['o'].notna() & mg['n'].notna()).sum()),
                     '추가': ', '.join(f"{a[:2]}{b} {c}" for a, b, c in mg.loc[mg['o'].isna(), key].itertuples(index=False)),
                     '빠짐': ', '.join(f"{a[:2]}{b} {c}" for a, b, c in mg.loc[mg['n'].isna(), key].itertuples(index=False))})
    CAND_CMP = pd.DataFrame(rows).set_index('V4')
    display(CAND_CMP)
    with pd.ExcelWriter(OUT_CMP / 'GPU_CPU비교_요약.xlsx') as xw:
        B_CMP.to_excel(xw, sheet_name='B비교'); CAND_CMP.to_excel(xw, sheet_name='후보비교')
    print('저장:', OUT_CMP / 'GPU_CPU비교_요약.xlsx')


## 읽는 법

| 결과 | 뜻 |
|---|---|
| `원래 vs GPU_1`에서 B 완전히 같은 칸 ≈ 100%, 후보 43개 | **원인은 GPU/CPU 차이가 맞음.** 원래 V4는 GPU로 만든 것 |
| GPU도 원래와 다르고 CPU와도 다름 | GPU/CPU만의 문제가 아님 → **xgboost 버전**이 원래와 다를 가능성. 원래 V4를 만든 노트북의 출력에서 xgboost 버전을 확인 |
| `GPU_1 vs GPU_2`가 다름 | GPU 학습이 매번 달라짐 → GPU로는 재현이 안 되므로 CPU로 고정해야 함 |
| 입력 일치가 False | 기상·특보 파일이 원래와 다름 (B 차이 전에 이것부터 맞춰야 함) |

CPU 결과는 지난번 재현과 같으면 47개가 나와야 합니다.
